# Lab 2 — Streaming LM Pipeline

**Configuration:**
- Dataset: `wikitext-103-raw-v1` (raw text, 103M tokens)
- Tokenizer: `gpt2` (GPT2TokenizerFast)
- Block size: **512** tokens
- Batch size: **16**

This notebook demonstrates a streaming data pipeline using Hugging Face Datasets
and PyTorch `IterableDataset`. The pipeline tokenizes on the fly, concatenates tokens
with a rolling buffer, and yields fixed-length chunks for language model training.

In [1]:
!pip install datasets transformers torch

  Using cached pandas-3.0.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached fsspec-2024.2.0-py3-none-any.whl.metadata (6.8 kB)
Using cached fsspec-2024.2.0-py3-none-any.whl (170 kB)
Using cached pandas-3.0.1-cp312-cp312-macosx_11_0_arm64.whl (9.9 MB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2026.2.0
    Uninstalling fsspec-2026.2.0:
      Successfully uninstalled fsspec-2026.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2024.2.0 which is incompatible.
recordlinkage 0.16 requires pandas<3,>=1, but you have pandas 3.0.1 which is incompatible.
fastf1 3.4.2 requires pandas<3.0.0,>=1.4.1, but you have pandas 3.0.1 which is incompatible.
mlflow 2.16.2 requires pandas<3, but you have pandas 3.0.1 which is incompatible.
s3fs 2024.3.1 requires fsspec==2024

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import IterableDataset, DataLoader
import torch
import time
import math

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


## 1) Load the dataset in streaming mode

We use `wikitext-103-raw-v1` in streaming mode so the dataset is not fully loaded into RAM.
This is the raw text version (not pre-tokenized) with 103M tokens — much larger than wikitext-2.

In [3]:
stream_dataset = load_dataset(
    "wikitext",
    "wikitext-103-raw-v1",
    split="train",
    streaming=True
)

print("stream_dataset created (streaming=True)")

stream_dataset created (streaming=True)


## 2) Initialize the tokenizer

GPT-2 doesn't have a pad token by default, so we set `pad_token` to `eos_token`.

In [4]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer: {tokenizer.__class__.__name__}")
print(f"Vocab size: {tokenizer.vocab_size}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer: GPT2TokenizerFast
Vocab size: 50257


## 3) Tokenization (lazy / streaming)

We map tokenization lazily over the streaming dataset. No padding or truncation here —
we want raw token sequences so they can be concatenated across documents.

In [5]:
def tokenize_function(examples):
    return tokenizer(examples["text"])

tokenized_stream = stream_dataset.map(tokenize_function, batched=True)
print("Tokenization mapping created (lazy, streaming)")

Tokenization mapping created (lazy, streaming)


## 4) Rolling buffer to group tokens into fixed-length blocks

We use a rolling buffer because streaming datasets are iterators and we cannot look ahead.
Block size of 512 provides more context per sequence than smaller block sizes.

In [6]:
block_size = 512

def group_texts_streaming(dataset_iter, block_size):
    """Generator that yields fixed-length token chunks from a streaming tokenized dataset.
    Uses a rolling buffer to concatenate tokens across document boundaries.
    """
    buffer = []
    for example in dataset_iter:
        ids = example.get("input_ids") if isinstance(example, dict) else None
        if ids is None:
            try:
                ids = example["input_ids"]
            except Exception:
                continue
        buffer.extend(ids)
        while len(buffer) >= block_size:
            chunk = buffer[:block_size]
            buffer = buffer[block_size:]
            yield {"input_ids": chunk, "attention_mask": [1] * block_size}

print(f"Block size set to {block_size} tokens")

Block size set to 512 tokens


## 5) Wrap generator in a PyTorch IterableDataset

This allows us to use PyTorch's `DataLoader` with the streaming generator.

In [7]:
class StreamingLMIterableDataset(IterableDataset):
    def __init__(self, hf_iterable_dataset, block_size):
        self.dataset = hf_iterable_dataset
        self.block_size = block_size

    def __iter__(self):
        return group_texts_streaming(self.dataset, self.block_size)

grouped_iterable_dataset = StreamingLMIterableDataset(tokenized_stream, block_size)
print("Wrapped grouped generator into StreamingLMIterableDataset")

Wrapped grouped generator into StreamingLMIterableDataset


## 6) Collate function and DataLoader

We produce tensors for `input_ids`, `attention_mask`, and `labels` (copy of input_ids for LM loss).

In [8]:
def collate_fn(batch):
    input_ids = torch.tensor([ex["input_ids"] for ex in batch], dtype=torch.long)
    attention_mask = torch.tensor([ex["attention_mask"] for ex in batch], dtype=torch.long)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": input_ids.clone()
    }

batch_size = 16
train_loader = DataLoader(grouped_iterable_dataset, batch_size=batch_size, collate_fn=collate_fn)
print(f"DataLoader created (streaming, batch_size={batch_size})")

DataLoader created (streaming, batch_size=16)


## 7) Inspect streaming batches and measure throughput

Iterate a few batches and report shapes and processing speed.

In [9]:
num_batches = 3
t0 = time.time()
for i, batch in enumerate(train_loader):
    print(f"Batch {i} -> input_ids shape: {batch['input_ids'].shape}, labels shape: {batch['labels'].shape}")
    if i + 1 >= num_batches:
        break
t1 = time.time()
elapsed = t1 - t0
processed_tokens = num_batches * batch_size * block_size
print(f"\nElapsed: {elapsed:.2f}s")
print(f"Processed tokens: {processed_tokens:,}")
print(f"Throughput: {processed_tokens/elapsed:,.0f} tokens/sec")

Batch 0 -> input_ids shape: torch.Size([16, 512]), labels shape: torch.Size([16, 512])
Batch 1 -> input_ids shape: torch.Size([16, 512]), labels shape: torch.Size([16, 512])
Batch 2 -> input_ids shape: torch.Size([16, 512]), labels shape: torch.Size([16, 512])

Elapsed: 1.19s
Processed tokens: 24,576
Throughput: 20,726 tokens/sec


## 8) Token statistics utility

Compute streaming statistics like average tokens per example for the first N examples.

In [10]:
def sample_stats(hf_iter, n_examples=200):
    """Compute token statistics over first n_examples from a streaming dataset."""
    cnt = 0
    token_counts = []
    t0 = time.time()
    for ex in hf_iter:
        ids = ex.get("input_ids") if isinstance(ex, dict) else None
        if ids is None:
            continue
        token_counts.append(len(ids))
        cnt += 1
        if cnt >= n_examples:
            break
    t1 = time.time()
    if token_counts:
        return {
            "examples_sampled": cnt,
            "avg_tokens_per_example": round(sum(token_counts) / len(token_counts), 2),
            "median_tokens_per_example": sorted(token_counts)[len(token_counts) // 2],
            "min_tokens": min(token_counts),
            "max_tokens": max(token_counts),
            "time_sec": round(t1 - t0, 2),
            "examples_per_sec": round(cnt / (t1 - t0), 2) if (t1 - t0) > 0 else float('inf')
        }
    return {}

# Re-create streaming dataset for stats (iterators are consumed after use)
stats_stream = load_dataset("wikitext", "wikitext-103-raw-v1", split="train", streaming=True)
stats_tokenized = stats_stream.map(tokenize_function, batched=True)
stats = sample_stats(stats_tokenized, n_examples=200)
for k, v in stats.items():
    print(f"  {k}: {v}")

  examples_sampled: 200
  avg_tokens_per_example: 58.98
  median_tokens_per_example: 13
  min_tokens: 0
  max_tokens: 351
  time_sec: 0.53
  examples_per_sec: 378.22


## 9) Notes

- This lab uses `wikitext-103-raw-v1` (raw text, 103M tokens) in streaming mode
- Block size of 512 and batch size of 16 provide larger context windows
- The pipeline is fully streaming and memory-efficient
- If streaming fails due to HF rate limits, switch to a smaller dataset or enable HF caching

### How to run
1. Run all cells in order in Jupyter / VS Code / Colab
2. Requires `datasets`, `transformers`, and `torch` installed